In [4]:
pip install PyQt5 pandas numpy scikit-learn matplotlib scipy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Final Code Updated

In [1]:
!/usr/bin/env python3
"""
Docking Validator Pro — Q1-ready decoy discrimination GUI

Designed for AutoDock Vina active/putative-active vs decoy score analysis.
Two score CSV files are required:
    putative actives: Ligand, Affinity (kcal/mol)
    decoys: Ligand, Affinity (kcal/mol)

Two separate structure CSV files can optionally be loaded for full decoy-quality analysis:
    putative actives structures: compound_name, smiles
    decoy structures: compound_name, smiles

The program joins structure files to docking-score files by compound identifier, so row order does not matter.
SMILES can also be present directly in the score CSV files.

Optional columns enable additional analyses:
    Actives: lead_id / Ligand, smiles, docking_score / Affinity (kcal/mol)
    Decoys: decoy_id / Ligand, parent_lead_id, smiles, docking_score / Affinity (kcal/mol)

Core analyses
-------------
- ROC-AUC with reproducible stratified bootstrap 95% CI
- Precision-recall curve and Average Precision
- EF at 1%, 5%, and 10% with actual top-N hit counts
- Standard BEDROC (alpha=20) via RDKit CalcBEDROC when available,
  with an exact mathematical fallback
- One-sided Mann-Whitney U, Welch's t-test, two-sample KS test
- Cohen's d with pooled sample SD
- Global active-rank summaries
- Per-lead matched-decoy analysis when parent mapping is available
  (explicit parent_lead_id or inferable lead-prefixed decoy IDs)
- Benjamini-Hochberg FDR for per-lead empirical p-values
- Optional physicochemical decoy-quality and ECFP4/Tanimoto checks when SMILES exist
- Publication-friendly export package

Interpretation note
-------------------
When the putative actives are selected from the same docking screen being
assessed, this should be described as an internal docking-score discrimination
assessment, not fully independent validation.
"""

from __future__ import annotations

import math
import os
import re
import traceback
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.metrics import (
    average_precision_score,
    auc,
    precision_recall_curve,
    roc_auc_score,
    roc_curve,
)

# Optional RDKit support: standard BEDROC + decoy-quality analysis.
RDKIT_AVAILABLE = False
try:
    from rdkit import Chem, DataStructs
    from rdkit.Chem import Crippen, Descriptors, Lipinski, rdFingerprintGenerator
    from rdkit.ML.Scoring.Scoring import CalcBEDROC
    RDKIT_AVAILABLE = True
except Exception:
    Chem = DataStructs = Crippen = Descriptors = Lipinski = rdFingerprintGenerator = None
    CalcBEDROC = None


# -----------------------------------------------------------------------------
# Core helpers
# -----------------------------------------------------------------------------

ID_CANDIDATES = [
    "Ligand", "ligand", "lead_id", "decoy_id", "compound_id", "compound_name", "Compound", "Name", "ID", "id"
]
SCORE_CANDIDATES = [
    "Affinity (kcal/mol)", "Affinity", "affinity", "docking_score", "Docking Score",
    "score", "binding_energy", "vina_score", "Vina Score"
]
SMILES_CANDIDATES = ["smiles", "SMILES", "canonical_smiles", "Canonical SMILES"]
PARENT_CANDIDATES = ["parent_lead_id", "parent_id", "parent", "lead_id"]


def _first_existing(columns: Sequence[str], candidates: Sequence[str]) -> Optional[str]:
    cols = list(columns)
    for c in candidates:
        if c in cols:
            return c
    lower = {str(c).lower(): c for c in cols}
    for c in candidates:
        if c.lower() in lower:
            return lower[c.lower()]
    return None


def detect_id_column(df: pd.DataFrame) -> Optional[str]:
    return _first_existing(df.columns, ID_CANDIDATES)


def detect_score_column(df: pd.DataFrame) -> Optional[str]:
    direct = _first_existing(df.columns, SCORE_CANDIDATES)
    if direct:
        return direct
    # Prefer columns whose names suggest docking scores.
    for c in df.columns:
        name = str(c).lower()
        if any(k in name for k in ("affinity", "score", "energy", "kcal")):
            return c
    # Last-resort: first mostly-numeric non-ID column.
    for c in df.columns:
        vals = pd.to_numeric(df[c], errors="coerce")
        if vals.notna().mean() >= 0.8:
            return c
    return None


def detect_smiles_column(df: pd.DataFrame) -> Optional[str]:
    return _first_existing(df.columns, SMILES_CANDIDATES)


def detect_parent_column(df: pd.DataFrame) -> Optional[str]:
    return _first_existing(df.columns, PARENT_CANDIDATES)


def clean_score_table(
    df: pd.DataFrame,
    label: int,
    id_col: str,
    score_col: str,
    smiles_col: Optional[str] = None,
    parent_col: Optional[str] = None,
) -> pd.DataFrame:
    if id_col not in df.columns:
        raise ValueError(f"ID column '{id_col}' not found.")
    if score_col not in df.columns:
        raise ValueError(f"Score column '{score_col}' not found.")

    out = pd.DataFrame({
        "compound_id": df[id_col].astype(str).str.strip(),
        "docking_score": pd.to_numeric(df[score_col], errors="coerce"),
        "label": int(label),
    })
    if smiles_col and smiles_col in df.columns:
        out["smiles"] = df[smiles_col].astype(str).str.strip()
    if parent_col and parent_col in df.columns:
        out["parent_lead_id"] = df[parent_col].astype(str).str.strip()

    before = len(out)
    finite = np.isfinite(out["docking_score"].to_numpy(dtype=float, na_value=np.nan))
    out = out.loc[finite].copy()
    out = out[out["compound_id"].str.len() > 0].copy()
    dup = out["compound_id"].duplicated(keep=False)
    if dup.any():
        ids = out.loc[dup, "compound_id"].astype(str).unique().tolist()
        raise ValueError(
            "Duplicate compound IDs were found in a docking-score file. "
            "Use one final docking score per compound before validation. Examples: "
            + ", ".join(ids[:10])
        )
    out.attrs["n_removed_nonfinite"] = before - len(out)
    return out.reset_index(drop=True)


def attach_smiles_table(
    score_table: pd.DataFrame,
    structure_df: pd.DataFrame,
    *,
    structure_id_col: Optional[str] = None,
    structure_smiles_col: Optional[str] = None,
    dataset_name: str = "structure",
) -> Tuple[pd.DataFrame, Dict[str, object]]:
    """Attach SMILES to an already-cleaned score table by exact compound identifier.

    Row order is ignored. Duplicate structure IDs are accepted only when they map to
    one unique SMILES; conflicting duplicate IDs raise an explicit error.
    """
    if structure_df is None or structure_df.empty:
        return score_table.copy(), {
            "provided": False, "matched": 0, "total": len(score_table), "coverage": 0.0,
            "id_col": None, "smiles_col": None, "unmatched_ids": list(score_table["compound_id"]),
        }

    structure_id_col = structure_id_col or detect_id_column(structure_df)
    structure_smiles_col = structure_smiles_col or detect_smiles_column(structure_df)
    if not structure_id_col or not structure_smiles_col:
        raise ValueError(
            f"Could not detect compound-ID and SMILES columns in {dataset_name} file. "
            f"Columns found: {list(structure_df.columns)}"
        )

    st = structure_df[[structure_id_col, structure_smiles_col]].copy()
    st.columns = ["compound_id", "smiles"]
    st["compound_id"] = st["compound_id"].astype(str).str.strip()
    st["smiles"] = st["smiles"].astype(str).str.strip()
    st = st[(st["compound_id"].str.len() > 0) & (st["smiles"].str.len() > 0)].copy()

    conflicts = (st.groupby("compound_id")["smiles"].nunique(dropna=True) > 1)
    conflict_ids = conflicts[conflicts].index.tolist()
    if conflict_ids:
        raise ValueError(
            f"{dataset_name} contains compound IDs mapped to more than one SMILES: "
            + ", ".join(map(str, conflict_ids[:10]))
        )
    st = st.drop_duplicates("compound_id", keep="first")

    out = score_table.copy()
    if "smiles" in out.columns:
        out = out.drop(columns=["smiles"])
    out = out.merge(st, on="compound_id", how="left", validate="many_to_one")
    valid_smiles = out["smiles"].notna() & out["smiles"].astype(str).str.strip().ne("")
    unmatched = out.loc[~valid_smiles, "compound_id"].astype(str).tolist()
    meta = {
        "provided": True,
        "matched": int(valid_smiles.sum()),
        "total": int(len(out)),
        "coverage": float(valid_smiles.mean()) if len(out) else 0.0,
        "id_col": structure_id_col,
        "smiles_col": structure_smiles_col,
        "unmatched_ids": unmatched,
    }
    return out, meta


def orient_scores(scores: Sequence[float], lower_is_better: bool = True) -> np.ndarray:
    s = np.asarray(scores, dtype=float)
    return -s if lower_is_better else s


def benjamini_hochberg(p_values: Sequence[float]) -> np.ndarray:
    p = np.asarray(p_values, dtype=float)
    q = np.full(len(p), np.nan, dtype=float)
    valid = np.isfinite(p)
    if not valid.any():
        return q
    pv = p[valid]
    order = np.argsort(pv)
    ranked = pv[order]
    m = len(ranked)
    adj = ranked * m / np.arange(1, m + 1)
    adj = np.minimum.accumulate(adj[::-1])[::-1]
    adj = np.clip(adj, 0.0, 1.0)
    unsorted = np.empty(m, dtype=float)
    unsorted[order] = adj
    q[valid] = unsorted
    return q


# -----------------------------------------------------------------------------
# ROC / PR / bootstrap
# -----------------------------------------------------------------------------

def stratified_bootstrap_indices(labels: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    labels = np.asarray(labels, dtype=int)
    a = np.flatnonzero(labels == 1)
    d = np.flatnonzero(labels == 0)
    if len(a) == 0 or len(d) == 0:
        raise ValueError("Both active and decoy classes are required.")
    return np.concatenate([
        rng.choice(a, size=len(a), replace=True),
        rng.choice(d, size=len(d), replace=True),
    ])


def roc_auc_with_bootstrap_ci(
    labels: Sequence[int],
    scores: Sequence[float],
    lower_is_better: bool = True,
    n_bootstraps: int = 5000,
    confidence_level: float = 0.95,
    seed: int = 42,
) -> Dict[str, object]:
    y = np.asarray(labels, dtype=int)
    raw = np.asarray(scores, dtype=float)
    finite = np.isfinite(raw) & np.isin(y, [0, 1])
    y, raw = y[finite], raw[finite]
    if len(np.unique(y)) != 2:
        raise ValueError("ROC-AUC requires both classes.")

    s = orient_scores(raw, lower_is_better)
    fpr, tpr, thresholds = roc_curve(y, s)
    roc_auc = float(roc_auc_score(y, s))

    rng = np.random.default_rng(seed)
    boots = np.empty(int(n_bootstraps), dtype=float)
    for i in range(int(n_bootstraps)):
        idx = stratified_bootstrap_indices(y, rng)
        boots[i] = roc_auc_score(y[idx], s[idx])

    alpha = 1.0 - confidence_level
    lo = float(np.quantile(boots, alpha / 2.0))
    hi = float(np.quantile(boots, 1.0 - alpha / 2.0))

    # Youden J is retained as a secondary diagnostic, not a primary endpoint.
    j = tpr - fpr
    best = int(np.nanargmax(j))
    sensitivity = float(tpr[best])
    specificity = float(1.0 - fpr[best])

    return {
        "roc_auc": roc_auc,
        "ci_lower": lo,
        "ci_upper": hi,
        "fpr": fpr,
        "tpr": tpr,
        "thresholds": thresholds,
        "sensitivity_youden": sensitivity,
        "specificity_youden": specificity,
        "bootstrap_aucs": boots,
    }


def roc_bootstrap_band(
    labels: Sequence[int],
    scores: Sequence[float],
    lower_is_better: bool = True,
    n_bootstraps: int = 1000,
    confidence_level: float = 0.95,
    seed: int = 42,
    grid_points: int = 201,
) -> Dict[str, np.ndarray]:
    y = np.asarray(labels, dtype=int)
    raw = np.asarray(scores, dtype=float)
    s = orient_scores(raw, lower_is_better)
    grid = np.linspace(0.0, 1.0, int(grid_points))
    rng = np.random.default_rng(seed)
    curves = []
    for _ in range(int(n_bootstraps)):
        idx = stratified_bootstrap_indices(y, rng)
        fpr, tpr, _ = roc_curve(y[idx], s[idx])
        interp = np.interp(grid, fpr, tpr)
        interp[0] = 0.0
        interp[-1] = 1.0
        curves.append(interp)
    mat = np.vstack(curves)
    alpha = 1.0 - confidence_level
    return {
        "fpr_grid": grid,
        "tpr_lower": np.quantile(mat, alpha / 2.0, axis=0),
        "tpr_upper": np.quantile(mat, 1.0 - alpha / 2.0, axis=0),
    }


def precision_recall_metrics(
    labels: Sequence[int], scores: Sequence[float], lower_is_better: bool = True
) -> Dict[str, object]:
    y = np.asarray(labels, dtype=int)
    s = orient_scores(scores, lower_is_better)
    precision, recall, thresholds = precision_recall_curve(y, s)
    ap = float(average_precision_score(y, s))
    prevalence = float(np.mean(y))
    return {
        "average_precision": ap,
        "precision": precision,
        "recall": recall,
        "thresholds": thresholds,
        "prevalence": prevalence,
    }


# -----------------------------------------------------------------------------
# Enrichment / BEDROC
# -----------------------------------------------------------------------------

def enrichment_metrics(
    labels: Sequence[int],
    scores: Sequence[float],
    fractions: Sequence[float] = (0.01, 0.05, 0.10),
    lower_is_better: bool = True,
) -> pd.DataFrame:
    y = np.asarray(labels, dtype=int)
    raw = np.asarray(scores, dtype=float)
    order = np.argsort(raw) if lower_is_better else np.argsort(raw)[::-1]
    ys = y[order]
    n = len(ys)
    a = int(ys.sum())
    if n == 0 or a == 0:
        raise ValueError("Enrichment analysis requires at least one active.")

    rows = []
    for frac in fractions:
        n_top = max(1, int(n * float(frac)))  # preserves the original analysis rule
        a_top = int(ys[:n_top].sum())
        random_expected = n_top * (a / n)
        ef = float(a_top / random_expected) if random_expected > 0 else np.nan
        rows.append({
            "fraction": float(frac),
            "percent": float(frac * 100),
            "n_top": n_top,
            "actives_top": a_top,
            "EF": ef,
            "hit_rate": float(a_top / n_top),
            "active_recovery": float(a_top / a),
        })
    return pd.DataFrame(rows)


def bedroc_fallback(
    labels: Sequence[int], scores: Sequence[float], alpha: float = 20.0,
    lower_is_better: bool = True
) -> Tuple[float, float]:
    y = np.asarray(labels, dtype=int)
    raw = np.asarray(scores, dtype=float)
    if alpha <= 0:
        raise ValueError("BEDROC alpha must be > 0.")
    order = np.argsort(raw) if lower_is_better else np.argsort(raw)[::-1]
    ys = y[order]
    n = len(ys)
    n_a = int(ys.sum())
    if n == 0 or n_a == 0:
        return 0.0, 0.0
    if n_a == n:
        return 1.0, 1.0

    active_ranks = np.flatnonzero(ys == 1) + 1  # 1-based
    sum_exp = float(np.sum(np.exp(-alpha * active_ranks / n)))
    denom = (1.0 / n) * ((1.0 - math.exp(-alpha)) / (math.exp(alpha / n) - 1.0))
    rie = sum_exp / (n_a * denom)

    ratio = n_a / n
    rie_max = (1.0 - math.exp(-alpha * ratio)) / (ratio * (1.0 - math.exp(-alpha)))
    rie_min = (1.0 - math.exp(alpha * ratio)) / (ratio * (1.0 - math.exp(alpha)))
    bedroc = (rie - rie_min) / (rie_max - rie_min) if rie_max != rie_min else 1.0
    return float(bedroc), float(rie)


def standard_bedroc(
    labels: Sequence[int], scores: Sequence[float], alpha: float = 20.0,
    lower_is_better: bool = True
) -> Dict[str, float]:
    y = np.asarray(labels, dtype=int)
    raw = np.asarray(scores, dtype=float)
    order = np.argsort(raw) if lower_is_better else np.argsort(raw)[::-1]
    sorted_y = y[order]
    sorted_scores = raw[order]

    fallback, rie = bedroc_fallback(y, raw, alpha, lower_is_better)
    if RDKIT_AVAILABLE:
        score_matrix = [[float(s), int(lbl)] for s, lbl in zip(sorted_scores, sorted_y)]
        rdkit_val = float(CalcBEDROC(score_matrix, 1, float(alpha)))
        # Guard against implementation mistakes: both independent paths must agree.
        if not np.isclose(rdkit_val, fallback, atol=1e-12, rtol=1e-10):
            raise RuntimeError(
                f"BEDROC cross-check failed: RDKit={rdkit_val:.12f}, fallback={fallback:.12f}"
            )
        method = "RDKit CalcBEDROC + exact fallback cross-check"
        bedroc = rdkit_val
    else:
        bedroc = fallback
        method = "Exact Truchon-Bayly BEDROC fallback (RDKit unavailable)"
    return {"BEDROC": float(bedroc), "RIE": float(rie), "alpha": float(alpha), "method": method}


# -----------------------------------------------------------------------------
# Score-distribution statistics / ranks
# -----------------------------------------------------------------------------

def score_distribution_tests(active_scores: Sequence[float], decoy_scores: Sequence[float]) -> Dict[str, float]:
    a = np.asarray(active_scores, dtype=float)
    d = np.asarray(decoy_scores, dtype=float)
    a = a[np.isfinite(a)]
    d = d[np.isfinite(d)]
    if len(a) < 2 or len(d) < 2:
        raise ValueError("At least two valid scores per group are required.")

    mw = stats.mannwhitneyu(a, d, alternative="less")
    welch = stats.ttest_ind(a, d, equal_var=False)
    ks = stats.ks_2samp(a, d, alternative="two-sided", mode="auto")

    n1, n2 = len(a), len(d)
    v1, v2 = np.var(a, ddof=1), np.var(d, ddof=1)
    pooled = math.sqrt(((n1 - 1) * v1 + (n2 - 1) * v2) / (n1 + n2 - 2))
    cohen_d = float((np.mean(a) - np.mean(d)) / pooled) if pooled > 0 else np.nan

    return {
        "active_mean": float(np.mean(a)),
        "active_sd": float(np.std(a, ddof=1)),
        "decoy_mean": float(np.mean(d)),
        "decoy_sd": float(np.std(d, ddof=1)),
        "mannwhitney_u": float(mw.statistic),
        "mannwhitney_p_one_sided": float(mw.pvalue),
        "welch_t": float(welch.statistic),
        "welch_p_two_sided": float(welch.pvalue),
        "ks_statistic": float(ks.statistic),
        "ks_p_two_sided": float(ks.pvalue),
        "cohens_d": cohen_d,
    }


def global_rank_summary(labels: Sequence[int], scores: Sequence[float], lower_is_better: bool = True) -> Dict[str, float]:
    y = np.asarray(labels, dtype=int)
    raw = np.asarray(scores, dtype=float)
    order = np.argsort(raw) if lower_is_better else np.argsort(raw)[::-1]
    ys = y[order]
    ranks = np.flatnonzero(ys == 1) + 1
    if len(ranks) == 0:
        return {"best_active_rank": np.nan, "worst_active_rank": np.nan,
                "mean_active_rank": np.nan, "median_active_rank": np.nan}
    return {
        "best_active_rank": int(np.min(ranks)),
        "worst_active_rank": int(np.max(ranks)),
        "mean_active_rank": float(np.mean(ranks)),
        "median_active_rank": float(np.median(ranks)),
    }


# -----------------------------------------------------------------------------
# Matched-decoy mapping and analysis
# -----------------------------------------------------------------------------

def infer_parent_ids(decoy_ids: Sequence[str], lead_ids: Sequence[str]) -> Tuple[List[Optional[str]], float]:
    leads = sorted({str(x).strip() for x in lead_ids}, key=len, reverse=True)
    mapped: List[Optional[str]] = []
    for decoy in decoy_ids:
        did = str(decoy).strip()
        hit = None
        for lead in leads:
            if did == lead or did.startswith(lead + "_") or did.startswith(lead + "-"):
                hit = lead
                break
        if hit is None:
            # Common naming convention: <lead>_<integer>
            stripped = re.sub(r"[_-]\d+$", "", did)
            if stripped in leads:
                hit = stripped
        mapped.append(hit)
    coverage = float(np.mean([x is not None for x in mapped])) if mapped else 0.0
    return mapped, coverage


def matched_decoy_analysis(
    actives: pd.DataFrame,
    decoys: pd.DataFrame,
    lower_is_better: bool = True,
) -> Tuple[pd.DataFrame, Dict[str, object]]:
    d = decoys.copy()
    mapping_source = "none"
    if "parent_lead_id" in d.columns and d["parent_lead_id"].astype(str).str.strip().ne("").any():
        d["parent_lead_id"] = d["parent_lead_id"].astype(str).str.strip()
        mapping_source = "explicit parent_lead_id column"
        coverage = float(d["parent_lead_id"].isin(set(actives["compound_id"])).mean())
    else:
        mapped, coverage = infer_parent_ids(d["compound_id"], actives["compound_id"])
        d["parent_lead_id"] = mapped
        mapping_source = "inferred from decoy ID prefix"

    rows = []
    for _, ar in actives.iterrows():
        lead = str(ar["compound_id"])
        s = float(ar["docking_score"])
        ds = d.loc[d["parent_lead_id"] == lead, "docking_score"].to_numpy(dtype=float)
        ds = ds[np.isfinite(ds)]
        if len(ds) == 0:
            continue

        if lower_is_better:
            n_equal_or_better = int(np.sum(ds <= s))
            n_strictly_worse = int(np.sum(ds > s))
            n_ties = int(np.sum(ds == s))
        else:
            n_equal_or_better = int(np.sum(ds >= s))
            n_strictly_worse = int(np.sum(ds < s))
            n_ties = int(np.sum(ds == s))

        rank = 1 + n_equal_or_better
        pct_beaten = 100.0 * (n_strictly_worse + 0.5 * n_ties) / len(ds)
        p_emp = (1.0 + n_equal_or_better) / (len(ds) + 1.0)
        gap = s - float(np.median(ds))

        rows.append({
            "lead_id": lead,
            "lead_score": s,
            "n_matched_decoys": int(len(ds)),
            "lead_rank_within_matched_set": int(rank),
            "percent_matched_decoys_beaten": float(pct_beaten),
            "decoy_median": float(np.median(ds)),
            "decoy_q25": float(np.quantile(ds, 0.25)),
            "decoy_q75": float(np.quantile(ds, 0.75)),
            "decoy_p05": float(np.quantile(ds, 0.05)),
            "decoy_p95": float(np.quantile(ds, 0.95)),
            "lead_minus_decoy_median": float(gap),
            "empirical_p": float(p_emp),
        })

    result = pd.DataFrame(rows)
    if not result.empty:
        result["BH_FDR_q"] = benjamini_hochberg(result["empirical_p"].to_numpy())
        result = result.sort_values(["lead_rank_within_matched_set", "lead_score"], ascending=[True, True])
    meta = {
        "mapping_source": mapping_source,
        "mapping_coverage": coverage,
        "n_leads_analyzed": int(len(result)),
        "expected_leads": int(len(actives)),
    }
    return result.reset_index(drop=True), meta


# -----------------------------------------------------------------------------
# Optional decoy-quality validation when SMILES are available
# -----------------------------------------------------------------------------

def _mol_from_smiles(smiles: str):
    if not RDKIT_AVAILABLE:
        return None
    try:
        return Chem.MolFromSmiles(str(smiles))
    except Exception:
        return None


def molecular_properties(smiles: str) -> Optional[Dict[str, float]]:
    mol = _mol_from_smiles(smiles)
    if mol is None:
        return None
    return {
        "MW": float(Descriptors.MolWt(mol)),
        "LogP": float(Crippen.MolLogP(mol)),
        "TPSA": float(Descriptors.TPSA(mol)),
        "HBD": float(Lipinski.NumHDonors(mol)),
        "HBA": float(Lipinski.NumHAcceptors(mol)),
        "RotB": float(Lipinski.NumRotatableBonds(mol)),
        "FormalCharge": float(Chem.GetFormalCharge(mol)),
    }


def decoy_quality_analysis(actives: pd.DataFrame, decoys: pd.DataFrame) -> Dict[str, object]:
    if not RDKIT_AVAILABLE:
        return {"available": False, "message": "RDKit unavailable."}
    if "smiles" not in actives.columns or "smiles" not in decoys.columns:
        return {"available": False, "message": "SMILES columns are not present in both input files."}

    matched, meta = matched_decoy_analysis(actives, decoys, lower_is_better=True)
    # Ensure decoys have parent IDs using same inference path.
    d = decoys.copy()
    if "parent_lead_id" not in d.columns or not d["parent_lead_id"].astype(str).str.strip().ne("").any():
        mapped, coverage = infer_parent_ids(d["compound_id"], actives["compound_id"])
        d["parent_lead_id"] = mapped
    else:
        coverage = float(d["parent_lead_id"].isin(set(actives["compound_id"])).mean())

    act_prop_rows = []
    act_mols = {}
    invalid_active_smiles = 0
    invalid_decoy_smiles = 0
    for _, r in actives.iterrows():
        rec = molecular_properties(r["smiles"])
        mol = _mol_from_smiles(r["smiles"])
        if rec is None or mol is None:
            invalid_active_smiles += 1
            continue
        rec["lead_id"] = str(r["compound_id"])
        act_prop_rows.append(rec)
        act_mols[str(r["compound_id"])] = mol

    dec_prop_rows = []
    tan_rows = []
    fpgen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
    for _, r in d.iterrows():
        parent = str(r.get("parent_lead_id", ""))
        rec = molecular_properties(r["smiles"])
        mol = _mol_from_smiles(r["smiles"])
        if rec is None or mol is None:
            invalid_decoy_smiles += 1
        if rec is not None:
            rec["parent_lead_id"] = parent
            dec_prop_rows.append(rec)
        if mol is not None and parent in act_mols:
            fp_a = fpgen.GetFingerprint(act_mols[parent])
            fp_d = fpgen.GetFingerprint(mol)
            tan_rows.append({
                "parent_lead_id": parent,
                "decoy_id": str(r["compound_id"]),
                "ECFP4_Tanimoto": float(DataStructs.TanimotoSimilarity(fp_a, fp_d)),
            })

    act_props = pd.DataFrame(act_prop_rows)
    dec_props = pd.DataFrame(dec_prop_rows)
    if act_props.empty or dec_props.empty:
        return {"available": False, "message": "No valid SMILES were available for property analysis."}

    prop_names = ["MW", "LogP", "TPSA", "HBD", "HBA", "RotB", "FormalCharge"]
    parent_means = dec_props.groupby("parent_lead_id")[prop_names].mean().reset_index()
    paired = act_props.merge(parent_means, left_on="lead_id", right_on="parent_lead_id", suffixes=("_lead", "_decoy_mean"))

    balance_rows = []
    for p in prop_names:
        x = paired[f"{p}_lead"].to_numpy(dtype=float)
        y = paired[f"{p}_decoy_mean"].to_numpy(dtype=float)
        if len(x) >= 2 and len(y) >= 2:
            pooled = math.sqrt((np.var(x, ddof=1) + np.var(y, ddof=1)) / 2.0)
            smd = (np.mean(x) - np.mean(y)) / pooled if pooled > 0 else 0.0
        else:
            smd = np.nan
        balance_rows.append({
            "property": p,
            "lead_mean": float(np.mean(x)) if len(x) else np.nan,
            "matched_decoy_parent_mean": float(np.mean(y)) if len(y) else np.nan,
            "standardized_mean_difference": float(smd),
        })

    return {
        "available": True,
        "message": "OK",
        "mapping_coverage": coverage,
        "invalid_active_smiles": int(invalid_active_smiles),
        "invalid_decoy_smiles": int(invalid_decoy_smiles),
        "property_balance": pd.DataFrame(balance_rows),
        "tanimoto": pd.DataFrame(tan_rows),
    }


# -----------------------------------------------------------------------------
# Full analysis object
# -----------------------------------------------------------------------------

@dataclass
class AnalysisResult:
    actives: pd.DataFrame
    decoys: pd.DataFrame
    combined: pd.DataFrame
    roc: Dict[str, object]
    pr: Dict[str, object]
    enrichment: pd.DataFrame
    bedroc: Dict[str, float]
    statistics: Dict[str, float]
    ranks: Dict[str, float]
    matched: pd.DataFrame
    matched_meta: Dict[str, object]
    decoy_quality: Dict[str, object]
    settings: Dict[str, object] = field(default_factory=dict)

    def summary_metrics(self) -> pd.DataFrame:
        ef_map = {int(r.percent): r.EF for _, r in self.enrichment.iterrows()}
        hit_map = {int(r.percent): int(r.actives_top) for _, r in self.enrichment.iterrows()}
        top_map = {int(r.percent): int(r.n_top) for _, r in self.enrichment.iterrows()}
        rows = [
            ("N putative actives", len(self.actives)),
            ("N decoys", len(self.decoys)),
            ("ROC-AUC", self.roc["roc_auc"]),
            ("ROC-AUC 95% CI lower", self.roc["ci_lower"]),
            ("ROC-AUC 95% CI upper", self.roc["ci_upper"]),
            ("Average Precision", self.pr["average_precision"]),
            ("BEDROC", self.bedroc["BEDROC"]),
            ("BEDROC alpha", self.bedroc["alpha"]),
            ("EF1%", ef_map.get(1, np.nan)),
            ("EF1% active hits", hit_map.get(1, np.nan)),
            ("EF1% top N", top_map.get(1, np.nan)),
            ("EF5%", ef_map.get(5, np.nan)),
            ("EF5% active hits", hit_map.get(5, np.nan)),
            ("EF5% top N", top_map.get(5, np.nan)),
            ("EF10%", ef_map.get(10, np.nan)),
            ("EF10% active hits", hit_map.get(10, np.nan)),
            ("EF10% top N", top_map.get(10, np.nan)),
            ("Active mean score", self.statistics["active_mean"]),
            ("Active SD", self.statistics["active_sd"]),
            ("Decoy mean score", self.statistics["decoy_mean"]),
            ("Decoy SD", self.statistics["decoy_sd"]),
            ("Mann-Whitney one-sided p", self.statistics["mannwhitney_p_one_sided"]),
            ("Welch t-test two-sided p", self.statistics["welch_p_two_sided"]),
            ("KS two-sided p", self.statistics["ks_p_two_sided"]),
            ("Cohen's d", self.statistics["cohens_d"]),
            ("Best active rank", self.ranks["best_active_rank"]),
            ("Median active rank", self.ranks["median_active_rank"]),
            ("Worst active rank", self.ranks["worst_active_rank"]),
            ("Matched mapping coverage", self.matched_meta.get("mapping_coverage", np.nan)),
        ]
        return pd.DataFrame(rows, columns=["metric", "value"])


def run_analysis(
    actives_csv: str,
    decoys_csv: str,
    *,
    active_smiles_csv: Optional[str] = None,
    decoy_smiles_csv: Optional[str] = None,
    active_id_col: Optional[str] = None,
    active_score_col: Optional[str] = None,
    decoy_id_col: Optional[str] = None,
    decoy_score_col: Optional[str] = None,
    active_smiles_col: Optional[str] = None,
    decoy_smiles_col: Optional[str] = None,
    active_structure_id_col: Optional[str] = None,
    active_structure_smiles_col: Optional[str] = None,
    decoy_structure_id_col: Optional[str] = None,
    decoy_structure_smiles_col: Optional[str] = None,
    parent_col: Optional[str] = None,
    lower_is_better: bool = True,
    bedroc_alpha: float = 20.0,
    n_bootstraps: int = 5000,
    random_seed: int = 42,
) -> AnalysisResult:
    raw_a = pd.read_csv(actives_csv)
    raw_d = pd.read_csv(decoys_csv)

    active_id_col = active_id_col or detect_id_column(raw_a)
    active_score_col = active_score_col or detect_score_column(raw_a)
    decoy_id_col = decoy_id_col or detect_id_column(raw_d)
    decoy_score_col = decoy_score_col or detect_score_column(raw_d)
    active_smiles_col = active_smiles_col or detect_smiles_column(raw_a)
    decoy_smiles_col = decoy_smiles_col or detect_smiles_column(raw_d)
    parent_col = parent_col or detect_parent_column(raw_d)

    missing = [
        name for name, val in [
            ("active ID", active_id_col), ("active score", active_score_col),
            ("decoy ID", decoy_id_col), ("decoy score", decoy_score_col),
        ] if not val
    ]
    if missing:
        raise ValueError("Could not detect required columns: " + ", ".join(missing))

    a = clean_score_table(raw_a, 1, active_id_col, active_score_col, active_smiles_col)
    d = clean_score_table(raw_d, 0, decoy_id_col, decoy_score_col, decoy_smiles_col, parent_col)
    if len(a) == 0 or len(d) == 0:
        raise ValueError("No valid active or decoy scores remained after cleaning.")

    active_structure_meta = {"provided": False, "matched": 0, "total": len(a), "coverage": 0.0, "unmatched_ids": []}
    decoy_structure_meta = {"provided": False, "matched": 0, "total": len(d), "coverage": 0.0, "unmatched_ids": []}
    if active_smiles_csv:
        raw_as = pd.read_csv(active_smiles_csv)
        a, active_structure_meta = attach_smiles_table(
            a, raw_as, structure_id_col=active_structure_id_col,
            structure_smiles_col=active_structure_smiles_col, dataset_name="active structure"
        )
    elif "smiles" in a.columns:
        ok = a["smiles"].astype(str).str.strip().ne("")
        active_structure_meta = {"provided": True, "matched": int(ok.sum()), "total": len(a),
                                 "coverage": float(ok.mean()), "unmatched_ids": a.loc[~ok, "compound_id"].tolist()}

    if decoy_smiles_csv:
        raw_ds = pd.read_csv(decoy_smiles_csv)
        d, decoy_structure_meta = attach_smiles_table(
            d, raw_ds, structure_id_col=decoy_structure_id_col,
            structure_smiles_col=decoy_structure_smiles_col, dataset_name="decoy structure"
        )
    elif "smiles" in d.columns:
        ok = d["smiles"].astype(str).str.strip().ne("")
        decoy_structure_meta = {"provided": True, "matched": int(ok.sum()), "total": len(d),
                                "coverage": float(ok.mean()), "unmatched_ids": d.loc[~ok, "compound_id"].tolist()}

    combined = pd.concat([a, d], ignore_index=True, sort=False)
    labels = combined["label"].to_numpy(dtype=int)
    scores = combined["docking_score"].to_numpy(dtype=float)

    roc_res = roc_auc_with_bootstrap_ci(
        labels, scores, lower_is_better, n_bootstraps=n_bootstraps,
        confidence_level=0.95, seed=random_seed
    )
    pr_res = precision_recall_metrics(labels, scores, lower_is_better)
    ef = enrichment_metrics(labels, scores, (0.01, 0.05, 0.10), lower_is_better)
    bedroc = standard_bedroc(labels, scores, bedroc_alpha, lower_is_better)
    stats_res = score_distribution_tests(
        a["docking_score"].to_numpy(dtype=float), d["docking_score"].to_numpy(dtype=float)
    )
    rank_res = global_rank_summary(labels, scores, lower_is_better)
    matched, matched_meta = matched_decoy_analysis(a, d, lower_is_better)
    quality = decoy_quality_analysis(a, d)

    return AnalysisResult(
        actives=a, decoys=d, combined=combined, roc=roc_res, pr=pr_res,
        enrichment=ef, bedroc=bedroc, statistics=stats_res, ranks=rank_res,
        matched=matched, matched_meta=matched_meta, decoy_quality=quality,
        settings={
            "lower_is_better": lower_is_better,
            "bedroc_alpha": bedroc_alpha,
            "n_bootstraps": n_bootstraps,
            "random_seed": random_seed,
            "active_id_col": active_id_col,
            "active_score_col": active_score_col,
            "decoy_id_col": decoy_id_col,
            "decoy_score_col": decoy_score_col,
            "active_smiles_col": active_smiles_col,
            "decoy_smiles_col": decoy_smiles_col,
            "parent_col": parent_col,
            "active_structure_meta": active_structure_meta,
            "decoy_structure_meta": decoy_structure_meta,
            "active_smiles_csv": active_smiles_csv,
            "decoy_smiles_csv": decoy_smiles_csv,
        },
    )


# -----------------------------------------------------------------------------
# Export helpers
# -----------------------------------------------------------------------------

def _format_p(x: float) -> str:
    if not np.isfinite(x):
        return "NA"
    return f"{x:.3e}" if x < 0.001 else f"{x:.4f}"


def build_text_report(result: AnalysisResult) -> str:
    s = result.statistics
    ef = result.enrichment.set_index("percent")
    qmsg = result.decoy_quality.get("message", "")
    lines = [
        "DOCKING SCORE DISCRIMINATION REPORT",
        "=" * 40,
        "",
        "Interpretation scope:",
        "This benchmark assesses internal docking-score discrimination when the putative active set",
        "comes from the same docking screen. It should not be described as independent experimental validation.",
        "",
        f"Putative actives: {len(result.actives)}",
        f"Decoys: {len(result.decoys)}",
        f"ROC-AUC: {result.roc['roc_auc']:.6f} (95% stratified-bootstrap CI {result.roc['ci_lower']:.6f}-{result.roc['ci_upper']:.6f})",
        f"Average Precision: {result.pr['average_precision']:.6f}",
        f"BEDROC (alpha={result.bedroc['alpha']:.1f}): {result.bedroc['BEDROC']:.6f}",
        f"BEDROC implementation: {result.bedroc['method']}",
        "",
    ]
    for pct in (1, 5, 10):
        if pct in ef.index:
            r = ef.loc[pct]
            lines.append(f"EF{pct}%: {r['EF']:.6f} ({int(r['actives_top'])} actives in top {int(r['n_top'])})")
    lines += [
        "",
        f"Active scores: {s['active_mean']:.4f} ± {s['active_sd']:.4f} kcal/mol (sample SD)",
        f"Decoy scores: {s['decoy_mean']:.4f} ± {s['decoy_sd']:.4f} kcal/mol (sample SD)",
        f"One-sided Mann-Whitney p: {_format_p(s['mannwhitney_p_one_sided'])}",
        f"Welch two-sample t-test p: {_format_p(s['welch_p_two_sided'])}",
        f"Two-sample KS p: {_format_p(s['ks_p_two_sided'])}",
        f"Cohen's d: {s['cohens_d']:.4f}",
        f"Active ranks: best {result.ranks['best_active_rank']}, median {result.ranks['median_active_rank']:.1f}, worst {result.ranks['worst_active_rank']}",
        "",
        f"Matched-decoy mapping: {result.matched_meta.get('mapping_source')} ({100*result.matched_meta.get('mapping_coverage',0):.1f}% coverage)",
        f"Leads with matched-decoy analysis: {result.matched_meta.get('n_leads_analyzed',0)}/{result.matched_meta.get('expected_leads',0)}",
        f"Decoy-quality analysis: {qmsg}",
        f"Active structure merge coverage: {100*result.settings.get('active_structure_meta',{}).get('coverage',0):.1f}%",
        f"Decoy structure merge coverage: {100*result.settings.get('decoy_structure_meta',{}).get('coverage',0):.1f}%",
    ]
    if result.decoy_quality.get("available"):
        tan = result.decoy_quality["tanimoto"]["ECFP4_Tanimoto"]
        lines += [
            f"Valid structure analysis: {len(result.actives)-result.decoy_quality.get('invalid_active_smiles',0)}/{len(result.actives)} actives; "
            f"{len(result.decoys)-result.decoy_quality.get('invalid_decoy_smiles',0)}/{len(result.decoys)} decoys",
            f"ECFP4 Tanimoto: mean {tan.mean():.4f}, median {tan.median():.4f}, max {tan.max():.4f}",
            "Property-balance SMDs:",
        ]
        for _, row in result.decoy_quality["property_balance"].iterrows():
            lines.append(f"  {row['property']}: {row['standardized_mean_difference']:.4f}")
    return "\n".join(lines)


def export_analysis(result: AnalysisResult, out_dir: str) -> List[str]:
    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)
    written = []

    def save_df(df: pd.DataFrame, name: str):
        p = out / name
        df.to_csv(p, index=False)
        written.append(str(p))

    save_df(result.summary_metrics(), "summary_metrics.csv")
    save_df(result.enrichment, "enrichment_metrics.csv")
    if not result.matched.empty:
        save_df(result.matched, "matched_decoy_results.csv")

    ranked = result.combined.copy()
    ranked = ranked.sort_values("docking_score", ascending=result.settings["lower_is_better"]).reset_index(drop=True)
    ranked.insert(0, "global_rank", np.arange(1, len(ranked) + 1))
    save_df(ranked, "ranked_compounds.csv")

    roc_df = pd.DataFrame({"FPR": result.roc["fpr"], "TPR": result.roc["tpr"]})
    save_df(roc_df, "roc_curve.csv")
    pr_df = pd.DataFrame({"Recall": result.pr["recall"], "Precision": result.pr["precision"]})
    save_df(pr_df, "precision_recall_curve.csv")

    if result.decoy_quality.get("available"):
        save_df(result.decoy_quality["property_balance"], "decoy_property_balance.csv")
        save_df(result.decoy_quality["tanimoto"], "decoy_tanimoto.csv")

    report_path = out / "Docking_Discrimination_Report.txt"
    report_path.write_text(build_text_report(result), encoding="utf-8")
    written.append(str(report_path))

    import json, platform
    metadata = {
        "tool": "Docking Validator Pro — four-file scientific edition",
        "python": platform.python_version(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "bedroc_method": result.bedroc.get("method"),
        "settings": result.settings,
        "matched_meta": result.matched_meta,
        "decoy_quality_available": bool(result.decoy_quality.get("available")),
    }
    meta_path = out / "analysis_metadata.json"
    meta_path.write_text(json.dumps(metadata, indent=2, default=str), encoding="utf-8")
    written.append(str(meta_path))

    # Publication-style figures (no seaborn dependency).
    import matplotlib.pyplot as plt

    # ROC
    fig, ax = plt.subplots(figsize=(6.5, 5.5))
    ax.plot(result.roc["fpr"], result.roc["tpr"], lw=2,
            label=f"ROC-AUC = {result.roc['roc_auc']:.4f}")
    ax.plot([0, 1], [0, 1], "--", lw=1, label="Random")
    band = roc_bootstrap_band(
        result.combined["label"], result.combined["docking_score"],
        result.settings["lower_is_better"], n_bootstraps=1000,
        confidence_level=0.95, seed=result.settings["random_seed"]
    )
    ax.fill_between(band["fpr_grid"], band["tpr_lower"], band["tpr_upper"], alpha=0.18,
                    label="95% bootstrap band")
    ax.set_xlabel("False positive rate")
    ax.set_ylabel("True positive rate")
    ax.set_title("ROC discrimination")
    ax.legend()
    ax.grid(alpha=0.2)
    fig.tight_layout()
    for ext in ("png", "tiff", "pdf"):
        p = out / f"ROC_curve.{ext}"
        fig.savefig(p, dpi=600, bbox_inches="tight")
        written.append(str(p))
    plt.close(fig)

    # PR
    fig, ax = plt.subplots(figsize=(6.5, 5.5))
    ax.plot(result.pr["recall"], result.pr["precision"], lw=2,
            label=f"AP = {result.pr['average_precision']:.4f}")
    ax.axhline(result.pr["prevalence"], ls="--", lw=1, label="Class prevalence")
    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.set_title("Precision-recall discrimination")
    ax.legend()
    ax.grid(alpha=0.2)
    fig.tight_layout()
    for ext in ("png", "tiff", "pdf"):
        p = out / f"Precision_recall_curve.{ext}"
        fig.savefig(p, dpi=600, bbox_inches="tight")
        written.append(str(p))
    plt.close(fig)

    # Early-enrichment figure
    fig, ax = plt.subplots(figsize=(6.8, 5.5))
    x = np.arange(len(result.enrichment))
    vals = result.enrichment["EF"].to_numpy(dtype=float)
    bars = ax.bar(x, vals)
    ax.set_xticks(x)
    ax.set_xticklabels([f"EF{int(p)}%" for p in result.enrichment["percent"]])
    ax.set_ylabel("Enrichment factor")
    ax.set_title(f"Early enrichment (BEDROC, α={result.bedroc['alpha']:.0f}: {result.bedroc['BEDROC']:.3f})")
    for bar, (_, row) in zip(bars, result.enrichment.iterrows()):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                f"{row['EF']:.1f}\n({int(row['actives_top'])}/{int(row['n_top'])})",
                ha="center", va="bottom", fontsize=9)
    ax.grid(axis="y", alpha=0.2)
    fig.tight_layout()
    for ext in ("png", "tiff", "pdf"):
        pp = out / f"Enrichment_metrics.{ext}"
        fig.savefig(pp, dpi=600, bbox_inches="tight")
        written.append(str(pp))
    plt.close(fig)

    # Score distributions
    fig, ax = plt.subplots(figsize=(7, 5.5))
    ax.hist(result.decoys["docking_score"], bins=30, alpha=0.55, density=True, label="Decoys")
    ax.hist(result.actives["docking_score"], bins=min(10, len(result.actives)), alpha=0.65, density=True, label="Putative actives")
    ax.set_xlabel("AutoDock Vina score (kcal/mol)")
    ax.set_ylabel("Density")
    ax.set_title("Docking-score distributions")
    ax.legend()
    fig.tight_layout()
    for ext in ("png", "tiff", "pdf"):
        p = out / f"Score_distributions.{ext}"
        fig.savefig(p, dpi=600, bbox_inches="tight")
        written.append(str(p))
    plt.close(fig)

    # Group-wise score boxplot for a compact manuscript panel
    fig, ax = plt.subplots(figsize=(5.8, 5.5))
    bp = ax.boxplot([result.actives["docking_score"].to_numpy(dtype=float),
                     result.decoys["docking_score"].to_numpy(dtype=float)],
                    tick_labels=["Putative actives", "Decoys"], showmeans=True)
    ax.set_ylabel("AutoDock Vina score (kcal/mol)")
    ax.set_title("Docking-score separation")
    ax.grid(axis="y", alpha=0.2)
    fig.tight_layout()
    for ext in ("png", "tiff", "pdf"):
        pp = out / f"Score_boxplot.{ext}"
        fig.savefig(pp, dpi=600, bbox_inches="tight")
        written.append(str(pp))
    plt.close(fig)

    # Matched decoy plot if available
    if not result.matched.empty:
        m = result.matched.copy().sort_values("lead_score")
        y = np.arange(len(m))
        fig, ax = plt.subplots(figsize=(8, max(5.5, 0.45 * len(m) + 2)))
        ax.hlines(y, m["decoy_p05"], m["decoy_p95"], lw=2, label="Matched decoy 5th-95th percentile")
        ax.scatter(m["decoy_median"], y, marker="s", label="Matched decoy median")
        ax.scatter(m["lead_score"], y, marker="o", label="Putative active")
        ax.set_yticks(y)
        ax.set_yticklabels(m["lead_id"])
        ax.set_xlabel("AutoDock Vina score (kcal/mol)")
        ax.set_title("Per-lead matched-decoy comparison")
        ax.legend()
        ax.grid(axis="x", alpha=0.2)
        fig.tight_layout()
        for ext in ("png", "tiff", "pdf"):
            p = out / f"Matched_decoy_comparison.{ext}"
            fig.savefig(p, dpi=600, bbox_inches="tight")
            written.append(str(p))
        plt.close(fig)

    # Decoy physicochemical-property balance
    if result.decoy_quality.get("available"):
        pb = result.decoy_quality["property_balance"].copy()
        fig, ax = plt.subplots(figsize=(7.5, 5.5))
        y = np.arange(len(pb))
        ax.barh(y, pb["standardized_mean_difference"])
        ax.axvline(0.0, lw=1)
        ax.axvline(0.1, ls="--", lw=1)
        ax.axvline(-0.1, ls="--", lw=1)
        ax.set_yticks(y)
        ax.set_yticklabels(pb["property"])
        ax.set_xlabel("Standardized mean difference (lead - matched-decoy mean)")
        ax.set_title("Physicochemical balance of matched decoys")
        ax.grid(axis="x", alpha=0.2)
        fig.tight_layout()
        for ext in ("png", "tiff", "pdf"):
            p = out / f"Decoy_property_balance.{ext}"
            fig.savefig(p, dpi=600, bbox_inches="tight")
            written.append(str(p))
        plt.close(fig)

        tan = result.decoy_quality["tanimoto"]["ECFP4_Tanimoto"].dropna().to_numpy(dtype=float)
        if len(tan):
            fig, ax = plt.subplots(figsize=(7.0, 5.5))
            ax.hist(tan, bins=25)
            ax.axvline(np.median(tan), ls="--", lw=1, label=f"Median = {np.median(tan):.3f}")
            ax.set_xlabel("Parent lead-decoy ECFP4 Tanimoto similarity")
            ax.set_ylabel("Count")
            ax.set_title("Structural dissimilarity of matched decoys")
            ax.legend(frameon=False)
            ax.grid(axis="y", alpha=0.2)
            fig.tight_layout()
            for ext in ("png", "tiff", "pdf"):
                p = out / f"Decoy_Tanimoto_distribution.{ext}"
                fig.savefig(p, dpi=600, bbox_inches="tight")
                written.append(str(p))
            plt.close(fig)

    return written


# =============================================================================
# FINAL PUBLICATION UPGRADE LAYER
# =============================================================================

import hashlib
import json
import platform
from datetime import datetime


def _standardized_mean_difference(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    x = x[np.isfinite(x)]
    y = y[np.isfinite(y)]
    if len(x) < 2 or len(y) < 2:
        return np.nan
    vx = np.var(x, ddof=1)
    vy = np.var(y, ddof=1)
    pooled = math.sqrt(((len(x)-1)*vx + (len(y)-1)*vy) / (len(x)+len(y)-2))
    if pooled == 0:
        return 0.0 if np.isclose(np.mean(x), np.mean(y)) else np.nan
    return float((np.mean(x)-np.mean(y))/pooled)


def _ensure_parent_mapping(actives: pd.DataFrame, decoys: pd.DataFrame):
    d = decoys.copy()
    if "parent_lead_id" in d.columns and d["parent_lead_id"].astype(str).str.strip().ne("").any():
        d["parent_lead_id"] = d["parent_lead_id"].astype(str).str.strip()
        source = "explicit parent_lead_id column"
        coverage = float(d["parent_lead_id"].isin(set(actives["compound_id"].astype(str))).mean())
    else:
        mapped, coverage = infer_parent_ids(d["compound_id"], actives["compound_id"])
        d["parent_lead_id"] = mapped
        source = "inferred from decoy ID prefix"
    return d, source, coverage


def matched_decoy_analysis_final(actives, decoys, lower_is_better=True):
    d, source, coverage = _ensure_parent_mapping(actives, decoys)
    rows = []
    for _, ar in actives.iterrows():
        lead = str(ar["compound_id"])
        lead_score = float(ar["docking_score"])
        ds = d.loc[d["parent_lead_id"] == lead, "docking_score"].to_numpy(dtype=float)
        ds = ds[np.isfinite(ds)]
        if len(ds) == 0:
            continue
        if lower_is_better:
            n_equal_or_better = int(np.sum(ds <= lead_score))
            n_worse = int(np.sum(ds > lead_score))
            n_ties = int(np.sum(ds == lead_score))
            favorable_advantage = float(np.median(ds) - lead_score)
        else:
            n_equal_or_better = int(np.sum(ds >= lead_score))
            n_worse = int(np.sum(ds < lead_score))
            n_ties = int(np.sum(ds == lead_score))
            favorable_advantage = float(lead_score - np.median(ds))
        rank = 1 + n_equal_or_better
        pct_beaten = 100.0 * (n_worse + 0.5*n_ties) / len(ds)
        p_emp = (1.0 + n_equal_or_better) / (len(ds) + 1.0)
        rows.append({
            "lead_id": lead,
            "lead_score": lead_score,
            "n_matched_decoys": int(len(ds)),
            "lead_rank_within_matched_set": int(rank),
            "percent_matched_decoys_beaten": float(pct_beaten),
            "decoy_median": float(np.median(ds)),
            "decoy_q25": float(np.quantile(ds, 0.25)),
            "decoy_q75": float(np.quantile(ds, 0.75)),
            "decoy_p05": float(np.quantile(ds, 0.05)),
            "decoy_p95": float(np.quantile(ds, 0.95)),
            "lead_minus_decoy_median": float(lead_score - np.median(ds)),
            "favorable_advantage": favorable_advantage,
            "empirical_p": float(p_emp),
        })
    out = pd.DataFrame(rows)
    if not out.empty:
        out["BH_FDR_q"] = benjamini_hochberg(out["empirical_p"].to_numpy())
        out = out.sort_values(["lead_rank_within_matched_set", "lead_score"], ascending=[True, True]).reset_index(drop=True)
    counts = d["parent_lead_id"].value_counts(dropna=False)
    lead_counts = {str(k): int(v) for k, v in counts.items() if pd.notna(k) and str(k) != "None"}
    meta = {
        "mapping_source": source,
        "mapping_coverage": coverage,
        "n_leads_analyzed": int(len(out)),
        "expected_leads": int(len(actives)),
        "decoys_per_lead": lead_counts,
        "min_decoys_per_lead": int(min(lead_counts.values())) if lead_counts else 0,
        "max_decoys_per_lead": int(max(lead_counts.values())) if lead_counts else 0,
    }
    return out, meta, d


def decoy_quality_analysis_final(actives, decoys):
    if not RDKIT_AVAILABLE:
        return {"available": False, "message": "RDKit unavailable."}
    if "smiles" not in actives.columns or "smiles" not in decoys.columns:
        return {"available": False, "message": "SMILES columns are not present in both input files."}
    d, source, coverage = _ensure_parent_mapping(actives, decoys)
    props = ["MW", "LogP", "TPSA", "HBD", "HBA", "RotB", "FormalCharge"]
    active_rows, decoy_rows, tan_rows = [], [], []
    active_mols = {}
    invalid_a = invalid_d = 0
    for _, r in actives.iterrows():
        rec = molecular_properties(r["smiles"])
        mol = _mol_from_smiles(r["smiles"])
        if rec is None or mol is None:
            invalid_a += 1
            continue
        rec["lead_id"] = str(r["compound_id"])
        active_rows.append(rec)
        active_mols[str(r["compound_id"])] = mol
    fpgen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
    for _, r in d.iterrows():
        parent = str(r.get("parent_lead_id", ""))
        rec = molecular_properties(r["smiles"])
        mol = _mol_from_smiles(r["smiles"])
        if rec is None or mol is None:
            invalid_d += 1
        if rec is not None:
            rec["parent_lead_id"] = parent
            rec["decoy_id"] = str(r["compound_id"])
            decoy_rows.append(rec)
        if mol is not None and parent in active_mols:
            tan_rows.append({
                "parent_lead_id": parent,
                "decoy_id": str(r["compound_id"]),
                "ECFP4_Tanimoto": float(DataStructs.TanimotoSimilarity(
                    fpgen.GetFingerprint(active_mols[parent]), fpgen.GetFingerprint(mol)))
            })
    aprops, dprops = pd.DataFrame(active_rows), pd.DataFrame(decoy_rows)
    if aprops.empty or dprops.empty:
        return {"available": False, "message": "No valid SMILES were available for property analysis."}
    parent_means = dprops.groupby("parent_lead_id")[props].mean().reset_index()
    paired = aprops.merge(parent_means, left_on="lead_id", right_on="parent_lead_id", suffixes=("_lead", "_decoy_mean"))
    matched_rows, pooled_rows = [], []
    for p in props:
        x = paired[f"{p}_lead"].to_numpy(float)
        y = paired[f"{p}_decoy_mean"].to_numpy(float)
        matched_rows.append({
            "property": p,
            "lead_mean": float(np.mean(x)),
            "matched_decoy_parent_mean": float(np.mean(y)),
            "standardized_mean_difference": _standardized_mean_difference(x, y),
        })
        xp = aprops[p].to_numpy(float)
        yp = dprops[p].to_numpy(float)
        pooled_rows.append({
            "property": p,
            "lead_mean": float(np.mean(xp)),
            "pooled_decoy_mean": float(np.mean(yp)),
            "standardized_mean_difference": _standardized_mean_difference(xp, yp),
        })
    return {
        "available": True,
        "message": "OK",
        "mapping_source": source,
        "mapping_coverage": coverage,
        "invalid_active_smiles": int(invalid_a),
        "invalid_decoy_smiles": int(invalid_d),
        "property_balance": pd.DataFrame(matched_rows),
        "matched_property_balance": pd.DataFrame(matched_rows),
        "pooled_property_balance": pd.DataFrame(pooled_rows),
        "tanimoto": pd.DataFrame(tan_rows),
    }


def validate_matched_design(result, expected_leads=10, expected_decoys_per_lead=50, strict=True):
    problems = []
    if len(result.actives) != int(expected_leads):
        problems.append(f"Expected {expected_leads} putative actives, found {len(result.actives)}.")
    if result.matched_meta.get("mapping_coverage", 0.0) < 1.0:
        problems.append(f"Parent mapping coverage is {100*result.matched_meta.get('mapping_coverage',0):.1f}%, not 100%.")
    counts = result.matched_meta.get("decoys_per_lead", {})
    for lead in result.actives["compound_id"].astype(str):
        n = int(counts.get(lead, 0))
        if n != int(expected_decoys_per_lead):
            problems.append(f"{lead}: expected {expected_decoys_per_lead} matched decoys, found {n}.")
    expected_total = int(expected_leads) * int(expected_decoys_per_lead)
    if len(result.decoys) != expected_total:
        problems.append(f"Expected {expected_total} decoys in total, found {len(result.decoys)}.")
    result.settings["matched_design_expected_leads"] = int(expected_leads)
    result.settings["matched_design_expected_decoys_per_lead"] = int(expected_decoys_per_lead)
    result.settings["matched_design_problems"] = problems
    if strict and problems:
        raise ValueError("Strict matched-decoy design validation failed:\n- " + "\n- ".join(problems))
    return problems


def run_analysis_final(*args, expected_leads=10, expected_decoys_per_lead=50, strict_matched_design=True, **kwargs):
    result = run_analysis(*args, **kwargs)
    matched, meta, mapped_decoys = matched_decoy_analysis_final(result.actives, result.decoys, result.settings["lower_is_better"])
    result.matched = matched
    result.matched_meta = meta
    result.decoys = mapped_decoys
    result.combined = pd.concat([result.actives, result.decoys], ignore_index=True, sort=False)
    result.decoy_quality = decoy_quality_analysis_final(result.actives, result.decoys)
    validate_matched_design(result, expected_leads, expected_decoys_per_lead, strict_matched_design)
    return result


def _save_figure_all(fig, out: Path, stem: str, written: list):
    for ext in ("png", "tiff", "pdf"):
        p = out / f"{stem}.{ext}"
        if ext == "pdf":
            fig.savefig(p, bbox_inches="tight", facecolor="white")
        else:
            fig.savefig(p, dpi=600, bbox_inches="tight", facecolor="white")
        written.append(str(p))


def _bold_publication_axis(ax, title_size=18, label_size=16, tick_size=14, legend_size=13,
                           spine_width=1.6, tick_width=1.4):
    """Apply bold, manuscript-readable styling to a Matplotlib axis."""
    title = ax.get_title()
    if title:
        ax.set_title(title, fontsize=title_size, fontweight="bold", pad=14)
    ax.xaxis.label.set_fontsize(label_size)
    ax.yaxis.label.set_fontsize(label_size)
    ax.xaxis.label.set_fontweight("bold")
    ax.yaxis.label.set_fontweight("bold")
    ax.tick_params(axis="both", which="major", labelsize=tick_size, width=tick_width, length=6)
    ax.tick_params(axis="both", which="minor", width=tick_width, length=3)
    for lab in ax.get_xticklabels() + ax.get_yticklabels():
        lab.set_fontweight("bold")
    ax.xaxis.get_offset_text().set_fontsize(tick_size)
    ax.yaxis.get_offset_text().set_fontsize(tick_size)
    ax.xaxis.get_offset_text().set_fontweight("bold")
    ax.yaxis.get_offset_text().set_fontweight("bold")
    for spine in ax.spines.values():
        spine.set_linewidth(spine_width)
    leg = ax.get_legend()
    if leg is not None:
        for t in leg.get_texts():
            t.set_fontsize(legend_size)
            t.set_fontweight("bold")
        if leg.get_title() is not None:
            leg.get_title().set_fontweight("bold")
            leg.get_title().set_fontsize(legend_size)


def _bold_annotation(text_obj, size=12):
    text_obj.set_fontsize(size)
    text_obj.set_fontweight("bold")
    return text_obj


def make_publication_figures(result):
    """Create bold, manuscript-readable publication figures.

    The score-distribution, enrichment-curve and BEDROC-profile styling follows
    the visual logic of the user's original validation notebook, while retaining
    the corrected scientific calculations from the final validation core.
    """
    import matplotlib.pyplot as plt
    from matplotlib import rc_context

    target = str(result.settings.get("target_name", "STAT3") or "STAT3").strip()

    # Large, bold typography is intentional so labels remain readable after
    # journal-column scaling. Curves/markers are also intentionally thick.
    pub_rc = {
        "font.size": 13,
        "font.weight": "bold",
        "axes.titleweight": "bold",
        "axes.labelweight": "bold",
        "axes.linewidth": 1.7,
        "xtick.major.width": 1.5,
        "ytick.major.width": 1.5,
        "xtick.minor.width": 1.2,
        "ytick.minor.width": 1.2,
        "legend.fontsize": 13,
        "lines.linewidth": 3.2,
        "savefig.facecolor": "white",
        "figure.facecolor": "white",
    }

    figs = {}
    with rc_context(pub_rc):
        # ------------------------------------------------------------------
        # ROC curve
        # ------------------------------------------------------------------
        fig, ax = plt.subplots(figsize=(7.4, 6.2))
        ax.plot(result.roc["fpr"], result.roc["tpr"], lw=3.4,
                label=f"ROC-AUC = {result.roc['roc_auc']:.4f}")
        ax.plot([0, 1], [0, 1], "--", lw=2.3, label="Random")
        band = roc_bootstrap_band(
            result.combined["label"], result.combined["docking_score"],
            result.settings["lower_is_better"],
            n_bootstraps=min(2000, int(result.settings["n_bootstraps"])),
            confidence_level=0.95, seed=result.settings["random_seed"]
        )
        ax.fill_between(band["fpr_grid"], band["tpr_lower"], band["tpr_upper"],
                        alpha=0.18, label="95% bootstrap band")
        ax.set_xlabel("False Positive Rate")
        ax.set_ylabel("True Positive Rate")
        ax.set_title(f"ROC Curve - {target}")
        ax.legend(frameon=True, framealpha=.96, loc="lower right")
        ax.grid(alpha=.25, linestyle="--", linewidth=1.0)
        ax.set_xlim(-0.02, 1.02); ax.set_ylim(-0.02, 1.02)
        _bold_publication_axis(ax)
        fig.tight_layout()
        figs["ROC_curve"] = fig

        # ------------------------------------------------------------------
        # Precision-recall curve
        # ------------------------------------------------------------------
        fig, ax = plt.subplots(figsize=(7.4, 6.2))
        ax.plot(result.pr["recall"], result.pr["precision"], lw=3.4,
                label=f"AP = {result.pr['average_precision']:.4f}")
        ax.axhline(result.pr["prevalence"], ls="--", lw=2.3,
                   label=f"Prevalence = {result.pr['prevalence']:.3f}")
        ax.set_xlabel("Recall")
        ax.set_ylabel("Precision")
        ax.set_title(f"Precision–Recall Curve - {target}")
        ax.legend(frameon=True, framealpha=.96)
        ax.grid(alpha=.25, linestyle="--", linewidth=1.0)
        _bold_publication_axis(ax)
        fig.tight_layout()
        figs["Precision_recall_curve"] = fig

        # ------------------------------------------------------------------
        # EF1/5/10 bar chart — distinct colors and sufficient headroom so
        # values never leave the plotting area.
        # ------------------------------------------------------------------
        fig, ax = plt.subplots(figsize=(7.6, 6.2))
        x = np.arange(len(result.enrichment))
        vals = result.enrichment["EF"].to_numpy(float)
        bar_colors = ["#2F6B9A", "#D8892B", "#4C956C"]
        bars = ax.bar(x, vals, color=bar_colors[:len(vals)],
                      edgecolor="black", linewidth=1.8, width=0.70)
        ax.set_xticks(x)
        ax.set_xticklabels([f"EF{int(p)}%" for p in result.enrichment["percent"]])
        ax.set_ylabel("Enrichment Factor")
        ax.set_title(f"Early Enrichment - {target} | BEDROC (α={result.bedroc['alpha']:.0f}) = {result.bedroc['BEDROC']:.3f}")
        ymax = max(1.0, float(np.nanmax(vals)))
        # Extra headroom keeps all EF and hit-count labels fully inside the axes.
        ax.set_ylim(0, ymax * 1.38)
        for bar, (_, row) in zip(bars, result.enrichment.iterrows()):
            y = bar.get_height() + ymax * 0.035
            t = ax.text(
                bar.get_x() + bar.get_width()/2, y,
                f"EF = {row['EF']:.1f}\nHits = {int(row['actives_top'])}/{int(row['n_top'])}",
                ha="center", va="bottom", clip_on=True,
                bbox=dict(boxstyle="round,pad=0.24", facecolor="white", edgecolor="black", linewidth=1.0, alpha=0.95)
            )
            _bold_annotation(t, 11.5)
        ax.grid(axis="y", alpha=.25, linestyle="--", linewidth=1.0)
        _bold_publication_axis(ax)
        fig.tight_layout()
        figs["Enrichment_metrics"] = fig

        # ------------------------------------------------------------------
        # Continuous enrichment curve — restored from the original notebook.
        # ------------------------------------------------------------------
        scores = result.combined["docking_score"].to_numpy(float)
        labels = result.combined["label"].to_numpy(int)
        if result.settings["lower_is_better"]:
            rank_scores = -scores
        else:
            rank_scores = scores
        order = np.argsort(rank_scores)[::-1]
        ranked_labels = labels[order]
        n_total = len(labels); n_actives = int(ranked_labels.sum())
        fractions = np.linspace(0.001, 0.20, 100)
        curve = []
        for frac in fractions:
            n_top = max(1, int(n_total * frac))
            a_top = int(ranked_labels[:n_top].sum())
            expected = n_top * (n_actives / n_total) if n_total else 0.0
            curve.append(a_top / expected if expected > 0 else 0.0)
        fig, ax = plt.subplots(figsize=(7.6, 6.2))
        ax.plot(fractions, curve, lw=3.5, color="#C63D3D", label="Enrichment")
        ax.axhline(1.0, color="black", ls="--", lw=2.2, alpha=.65, label="Random")
        ax.set_xlabel("Fraction of Database Screened")
        ax.set_ylabel("Enrichment Factor")
        ax.set_title(f"Enrichment Curve - {target}")
        ax.set_xlim(0, 0.20)
        max_curve = max(curve) if curve else 1.0
        ax.set_ylim(0, max(20.0, max_curve * 1.12))
        ax.legend(loc="upper right", frameon=True, framealpha=.96)
        ax.grid(alpha=.28, linestyle="--", linewidth=1.0)
        _bold_publication_axis(ax)
        fig.tight_layout()
        figs["Enrichment_curve"] = fig

        # ------------------------------------------------------------------
        # BEDROC early-recognition profile — visualization restored from the
        # original notebook, but the displayed BEDROC is the corrected
        # standard BEDROC from the final scientific core.
        # ------------------------------------------------------------------
        alpha = float(result.bedroc["alpha"])
        ranks = np.arange(1, n_total + 1, dtype=float)
        weights = np.exp(-alpha * ranks / n_total)
        hit_weights = weights * ranked_labels
        cumulative = np.cumsum(hit_weights)
        if cumulative.size and cumulative[-1] > 0:
            cumulative = cumulative / cumulative[-1]
        fig, ax = plt.subplots(figsize=(7.6, 6.2))
        ax.plot(ranks / n_total, cumulative, lw=3.5, color="#2E8B57",
                label=f"BEDROC (α={alpha:g}) = {result.bedroc['BEDROC']:.3f}")
        ax.set_xlabel("Fraction Screened")
        ax.set_ylabel("Normalized Early-Weighted Hit Recovery")
        ax.set_title(f"BEDROC Plot - {target}")
        ax.set_xlim(0, 1.0); ax.set_ylim(-0.02, 1.03)
        ax.legend(loc="lower right", frameon=True, framealpha=.96)
        ax.grid(alpha=.28, linestyle="--", linewidth=1.0)
        _bold_publication_axis(ax)
        fig.tight_layout()
        figs["BEDROC_plot"] = fig

        # ------------------------------------------------------------------
        # Score distribution — restored to the user's preferred original
        # design: common 30-bin range, blue putative actives and red decoys.
        # ------------------------------------------------------------------
        fig, ax = plt.subplots(figsize=(10.2, 6.2))
        active_scores = result.actives["docking_score"].to_numpy(float)
        decoy_scores = result.decoys["docking_score"].to_numpy(float)
        all_scores = np.concatenate([active_scores, decoy_scores])
        hist_range = (float(np.min(all_scores) - 1.0), float(np.max(all_scores) + 1.0))
        ax.hist(active_scores, bins=30, alpha=.70, density=True, range=hist_range,
                label="Putative Actives", color="blue", edgecolor="black", linewidth=.9)
        ax.hist(decoy_scores, bins=30, alpha=.70, density=True, range=hist_range,
                label="Decoys", color="red", edgecolor="black", linewidth=.9)
        direction = "lower is better" if result.settings["lower_is_better"] else "higher is better"
        ax.set_xlabel(f"Docking Score ({direction})")
        ax.set_ylabel("Density")
        ax.set_title(f"Score Distribution - {target}")
        ax.legend(frameon=True, framealpha=.96, loc="upper right")
        ax.grid(axis="y", alpha=.28, linestyle="--", linewidth=1.0)
        _bold_publication_axis(ax, title_size=18, label_size=16, tick_size=14, legend_size=14)
        fig.tight_layout()
        figs["Score_distributions"] = fig

        # ------------------------------------------------------------------
        # Score boxplot
        # ------------------------------------------------------------------
        fig, ax = plt.subplots(figsize=(6.5, 6.2))
        ax.boxplot(
            [active_scores, decoy_scores],
            tick_labels=["Putative Actives", "Decoys"], showmeans=True,
            boxprops={"linewidth": 2.0}, whiskerprops={"linewidth": 2.0},
            capprops={"linewidth": 2.0}, medianprops={"linewidth": 2.8},
            meanprops={"markersize": 8, "markeredgewidth": 1.5}
        )
        ax.set_ylabel("AutoDock Vina Score (kcal/mol)")
        ax.set_title("Docking-Score Separation")
        ax.grid(axis="y", alpha=.25, linestyle="--", linewidth=1.0)
        _bold_publication_axis(ax)
        fig.tight_layout()
        figs["Score_boxplot"] = fig

        # ------------------------------------------------------------------
        # Matched-decoy comparison
        # ------------------------------------------------------------------
        if not result.matched.empty:
            m = result.matched.copy().sort_values("lead_score", ascending=True).reset_index(drop=True)
            y = np.arange(len(m))
            fig, (ax1, ax2) = plt.subplots(
                1, 2, figsize=(15.2, max(6.4, .54*len(m)+2.8)),
                gridspec_kw={"width_ratios": [2.45, 1.1]}
            )
            ax1.hlines(y, m["decoy_p05"], m["decoy_p95"], lw=3.0,
                       label="Decoy 5th–95th Percentile")
            ax1.hlines(y, m["decoy_q25"], m["decoy_q75"], lw=8.0, alpha=.45,
                       label="Decoy IQR")
            ax1.scatter(m["decoy_median"], y, marker="s", s=72,
                        linewidths=1.3, edgecolors="black", label="Decoy Median", zorder=3)
            ax1.scatter(m["lead_score"], y, marker="D", s=82,
                        linewidths=1.3, edgecolors="black", label="Putative Active", zorder=4)
            ax1.set_yticks(y); ax1.set_yticklabels(m["lead_id"]); ax1.invert_yaxis()
            ax1.set_xlabel("AutoDock Vina Score (kcal/mol)")
            ax1.set_title("A. Lead Versus Matched Decoys")
            ax1.grid(axis="x", alpha=.25, linestyle="--", linewidth=1.0)
            _bold_publication_axis(ax1, title_size=17, label_size=15, tick_size=13, legend_size=12)

            bars = ax2.barh(y, m["percent_matched_decoys_beaten"],
                            color="#4C956C", edgecolor="black", linewidth=1.2)
            ax2.set_yticks(y); ax2.set_yticklabels([]); ax2.invert_yaxis(); ax2.set_xlim(0, 100)
            ax2.set_xlabel("Matched Decoys Beaten (%)"); ax2.set_title("B. Within-Set Ranking")
            ax2.grid(axis="x", alpha=.25, linestyle="--", linewidth=1.0)
            for bar, (_, row) in zip(bars, m.iterrows()):
                label = (f"{row['percent_matched_decoys_beaten']:.0f}% | "
                         f"rank {int(row['lead_rank_within_matched_set'])}/{int(row['n_matched_decoys'])+1}")
                t = ax2.text(min(97.0, max(12.0, bar.get_width()-2)),
                             bar.get_y()+bar.get_height()/2, label, ha="right", va="center")
                _bold_annotation(t, 10.5)
            _bold_publication_axis(ax2, title_size=17, label_size=15, tick_size=13, legend_size=12)

            handles, labels_leg = ax1.get_legend_handles_labels()
            leg = fig.legend(handles, labels_leg, loc="lower center", ncol=4, frameon=True,
                             framealpha=.95, bbox_to_anchor=(0.43, .012), fontsize=12)
            for t in leg.get_texts(): t.set_fontweight("bold")
            fig.tight_layout(rect=[0, .085, 1, 1])
            figs["Matched_decoy_comparison"] = fig

        # ------------------------------------------------------------------
        # Decoy physicochemical balance + structural dissimilarity
        # ------------------------------------------------------------------
        if result.decoy_quality.get("available"):
            mp = result.decoy_quality["matched_property_balance"]
            pp = result.decoy_quality["pooled_property_balance"]
            props = mp["property"].tolist(); y = np.arange(len(props)); h = .36
            fig, ax = plt.subplots(figsize=(9.2, 6.4))
            ax.barh(y-h/2, mp["standardized_mean_difference"], height=h,
                    color="#2F6B9A", label="Parent-Matched", edgecolor="black", linewidth=1.0)
            ax.barh(y+h/2, pp["standardized_mean_difference"], height=h,
                    color="#D8892B", label="Pooled", edgecolor="black", linewidth=1.0)
            ax.axvline(0, color="black", lw=2.0)
            ax.axvline(.1, color="#666666", ls="--", lw=2.0)
            ax.axvline(-.1, color="#666666", ls="--", lw=2.0)
            ax.set_yticks(y); ax.set_yticklabels(props)
            ax.set_xlabel("Standardized Mean Difference (Lead − Decoy)")
            ax.set_title("Physicochemical Balance of DUD-E Decoys")
            ax.legend(frameon=True, framealpha=.95)
            ax.grid(axis="x", alpha=.25, linestyle="--", linewidth=1.0)
            _bold_publication_axis(ax)
            fig.tight_layout(); figs["Decoy_property_balance"] = fig

            tan = result.decoy_quality["tanimoto"]["ECFP4_Tanimoto"].dropna().to_numpy(float)
            if len(tan):
                fig, ax = plt.subplots(figsize=(7.4, 6.2))
                ax.hist(tan, bins=25, color="#6F63A6", edgecolor="black", linewidth=1.2)
                ax.axvline(np.median(tan), color="#C63D3D", ls="--", lw=2.5,
                           label=f"Median = {np.median(tan):.3f}")
                ax.set_xlabel("Parent Lead–Decoy ECFP4 Tanimoto Similarity")
                ax.set_ylabel("Count")
                ax.set_title("Structural Dissimilarity of Matched Decoys")
                ax.legend(frameon=True, framealpha=.95)
                ax.grid(axis="y", alpha=.25, linestyle="--", linewidth=1.0)
                _bold_publication_axis(ax)
                fig.tight_layout(); figs["Decoy_Tanimoto_distribution"] = fig

        # ------------------------------------------------------------------
        # Four-panel global summary
        # ------------------------------------------------------------------
        fig, axes = plt.subplots(2, 2, figsize=(13.2, 10.2))
        ax = axes[0, 0]
        ax.plot(result.roc["fpr"], result.roc["tpr"], lw=3.0, label=f"AUC = {result.roc['roc_auc']:.4f}")
        ax.plot([0,1], [0,1], "--", lw=2.0, label="Random")
        ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
        ax.set_title("A. ROC Discrimination"); ax.legend(frameon=True, framealpha=.95); ax.grid(alpha=.25, linestyle="--")
        _bold_publication_axis(ax, title_size=15, label_size=13, tick_size=11, legend_size=10.5)

        ax = axes[0, 1]
        ax.plot(result.pr["recall"], result.pr["precision"], lw=3.0, label=f"AP = {result.pr['average_precision']:.4f}")
        ax.axhline(result.pr["prevalence"], ls="--", lw=2.0, label="Prevalence")
        ax.set_xlabel("Recall"); ax.set_ylabel("Precision"); ax.set_title("B. Precision–Recall")
        ax.legend(frameon=True, framealpha=.95); ax.grid(alpha=.25, linestyle="--")
        _bold_publication_axis(ax, title_size=15, label_size=13, tick_size=11, legend_size=10.5)

        ax = axes[1, 0]
        xx = np.arange(len(result.enrichment)); vals2 = result.enrichment["EF"].to_numpy(float)
        bars2 = ax.bar(xx, vals2, color=bar_colors[:len(vals2)], edgecolor="black", linewidth=1.3)
        ax.set_xticks(xx); ax.set_xticklabels([f"EF{int(p)}%" for p in result.enrichment["percent"]])
        ax.set_ylabel("Enrichment Factor"); ax.set_title(f"C. Early Enrichment | BEDROC = {result.bedroc['BEDROC']:.3f}")
        ymax2 = max(1.0, float(np.nanmax(vals2))); ax.set_ylim(0, ymax2*1.22); ax.grid(axis="y", alpha=.25, linestyle="--")
        for bar, (_, row) in zip(bars2, result.enrichment.iterrows()):
            t = ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+ymax2*.02,
                        f"{row['EF']:.1f}", ha="center", va="bottom", clip_on=True)
            _bold_annotation(t, 10.5)
        _bold_publication_axis(ax, title_size=15, label_size=13, tick_size=11, legend_size=10.5)

        ax = axes[1, 1]
        ax.boxplot([active_scores, decoy_scores], tick_labels=["Putative Actives", "Decoys"], showmeans=True,
                   boxprops={"linewidth": 1.8}, whiskerprops={"linewidth": 1.8}, capprops={"linewidth": 1.8},
                   medianprops={"linewidth": 2.5}, meanprops={"markersize": 7, "markeredgewidth": 1.3})
        ax.set_ylabel("AutoDock Vina Score (kcal/mol)"); ax.set_title("D. Score Separation"); ax.grid(axis="y", alpha=.25, linestyle="--")
        _bold_publication_axis(ax, title_size=15, label_size=13, tick_size=11, legend_size=10.5)

        fig.tight_layout(pad=2.0); figs["Global_discrimination_summary"] = fig

    return figs

def file_sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()


def export_analysis_final(result: AnalysisResult, out_dir: str, input_files=None):
    out = Path(out_dir); out.mkdir(parents=True, exist_ok=True)
    written = export_analysis(result, str(out))
    if result.decoy_quality.get("available"):
        p = out/"decoy_property_balance_parent_matched.csv"; result.decoy_quality["matched_property_balance"].to_csv(p,index=False); written.append(str(p))
        p = out/"decoy_property_balance_pooled.csv"; result.decoy_quality["pooled_property_balance"].to_csv(p,index=False); written.append(str(p))
    counts = pd.DataFrame([{"lead_id":k,"n_decoys":v} for k,v in result.matched_meta.get("decoys_per_lead",{}).items()])
    if not counts.empty:
        p = out/"matched_decoy_group_counts.csv"; counts.to_csv(p,index=False); written.append(str(p))
    figs = make_publication_figures(result)
    import matplotlib.pyplot as plt
    target = re.sub(r"[^A-Za-z0-9_.-]+", "_", str(result.settings.get("target_name", "STAT3") or "STAT3").strip())
    alias_map = {
        "Enrichment_curve": f"{target}_Enrichment_Curve",
        "BEDROC_plot": f"{target}_BEDROC_Plot",
        "Score_distributions": f"{target}_Score_Distribution",
    }
    for stem, fig in figs.items():
        _save_figure_all(fig,out,stem,written)
        if stem in alias_map:
            _save_figure_all(fig,out,alias_map[stem],written)
        plt.close(fig)
    versions = {"python":platform.python_version(),"numpy":np.__version__,"pandas":pd.__version__}
    try:
        import scipy, sklearn, matplotlib
        versions.update({"scipy":scipy.__version__,"scikit_learn":sklearn.__version__,"matplotlib":matplotlib.__version__})
    except Exception:
        pass
    if RDKIT_AVAILABLE:
        try:
            import rdkit; versions["rdkit"] = rdkit.__version__
        except Exception:
            versions["rdkit"] = "available"
    manifest = {"tool":"Docking Validator Pro — Final Publication Edition","generated_at":datetime.now().isoformat(timespec="seconds"),"versions":versions,"settings":result.settings,"bedroc_method":result.bedroc.get("method"),"matched_meta":result.matched_meta,"input_files":{}}
    if input_files:
        for label, path in input_files.items():
            if path and Path(path).exists():
                manifest["input_files"][label] = {"path":str(Path(path).resolve()),"sha256":file_sha256(path),"size_bytes":Path(path).stat().st_size}
    p = out/"Reproducibility_Manifest.json"; p.write_text(json.dumps(manifest,indent=2,default=str),encoding="utf-8"); written.append(str(p))
    return sorted(set(written))


def launch_gui_final():
    try:
        from PyQt5.QtCore import QThread, pyqtSignal, Qt
        from PyQt5.QtWidgets import QApplication,QMainWindow,QWidget,QVBoxLayout,QHBoxLayout,QGridLayout,QLabel,QPushButton,QFileDialog,QMessageBox,QTabWidget,QTextEdit,QTableWidget,QTableWidgetItem,QHeaderView,QGroupBox,QDoubleSpinBox,QSpinBox,QCheckBox,QLineEdit,QProgressBar,QFrame
        import matplotlib
        matplotlib.use("Qt5Agg")
        from matplotlib.backends.backend_qt5agg import FigureCanvasQTAgg as FigureCanvas
        from matplotlib.backends.backend_qt5agg import NavigationToolbar2QT as NavigationToolbar
        from matplotlib.figure import Figure
    except Exception as e:
        raise RuntimeError("PyQt5 is required for the GUI. Install with: python -m pip install PyQt5") from e

    class AnalysisWorker(QThread):
        completed = pyqtSignal(object)
        failed = pyqtSignal(str)
        def __init__(self, kwargs):
            super().__init__(); self.kwargs = kwargs
        def run(self):
            try:
                self.completed.emit(run_analysis_final(**self.kwargs))
            except Exception:
                self.failed.emit(traceback.format_exc())

    class PlotPanel(QWidget):
        def __init__(self):
            super().__init__(); layout=QVBoxLayout(self); self.canvas=FigureCanvas(Figure(figsize=(7,5))); self.toolbar=NavigationToolbar(self.canvas,self); layout.addWidget(self.toolbar); layout.addWidget(self.canvas)
        def show_figure(self, srcfig):
            import matplotlib.pyplot as plt, matplotlib.image as mpimg, tempfile
            self.canvas.figure.clear()
            tmp = Path(tempfile.gettempdir())/"_dvp_preview.png"
            srcfig.savefig(tmp,dpi=220,bbox_inches="tight",facecolor="white")
            ax=self.canvas.figure.add_subplot(111); ax.imshow(mpimg.imread(tmp)); ax.axis("off"); self.canvas.figure.tight_layout(); self.canvas.draw()
            try: tmp.unlink()
            except Exception: pass
            plt.close(srcfig)

    class App(QMainWindow):
        def __init__(self):
            super().__init__()
            self.setWindowTitle("Docking Protocol Validation Using Decoy Compounds")
            self.resize(1580, 960)
            self.setMinimumSize(1220, 790)
            self.paths={"actives_scores":"","decoys_scores":"","actives_smiles":"","decoys_smiles":""}
            self.result=None
            self.worker=None
            self.output_dir=""
            self.setStyleSheet("""
            QMainWindow{background:#eef3f8;}
            QWidget{font-family:'Segoe UI';font-size:10.5pt;color:#182230;}
            QGroupBox{font-weight:700;border:1px solid #d7e0ea;border-radius:10px;margin-top:10px;padding:13px 10px 10px 10px;background:white;}
            QGroupBox::title{subcontrol-origin:margin;left:14px;padding:0 6px;color:#17324d;}
            QPushButton{background:#245b9e;color:white;border:none;border-radius:7px;padding:9px 14px;font-weight:700;}
            QPushButton:hover{background:#1b4a83;}
            QPushButton#run{background:#14804a;} QPushButton#run:hover{background:#0f683c;}
            QPushButton#export{background:#6d43b5;} QPushButton#export:hover{background:#583595;}
            QPushButton#browseout{background:#475569;} QPushButton#browseout:hover{background:#334155;}
            QPushButton:disabled{background:#aeb8c5;color:#eef2f6;}
            QLineEdit,QSpinBox,QDoubleSpinBox{background:white;border:1px solid #cbd6e2;border-radius:6px;padding:7px;}
            QLineEdit:focus,QSpinBox:focus,QDoubleSpinBox:focus{border:1px solid #2e73b8;}
            QTabWidget::pane{border:1px solid #d7e0ea;border-radius:8px;background:white;}
            QTabBar::tab{padding:9px 14px;background:#e7edf4;margin-right:2px;border-top-left-radius:5px;border-top-right-radius:5px;}
            QTabBar::tab:selected{background:white;font-weight:700;color:#17324d;}
            QTableWidget{gridline-color:#e5ebf1;background:white;selection-background-color:#dcecff;}
            QHeaderView::section{background:#eef3f8;padding:7px;border:0;border-right:1px solid #d7e0ea;font-weight:700;}
            QTextEdit{background:white;border:0;}
            QProgressBar{border:1px solid #cbd6e2;border-radius:6px;text-align:center;background:white;}
            QProgressBar::chunk{background:#2e73b8;border-radius:5px;}
            """)
            root=QWidget(); self.setCentralWidget(root)
            main=QVBoxLayout(root); main.setContentsMargins(16,16,16,14); main.setSpacing(11)

            # Professional application header
            header=QFrame()
            header.setStyleSheet("QFrame{background:qlineargradient(x1:0,y1:0,x2:1,y2:0,stop:0 #0f2742,stop:0.58 #173b61,stop:1 #245b9e);border-radius:12px;}")
            hl=QVBoxLayout(header); hl.setContentsMargins(22,17,22,17); hl.setSpacing(5)
            title=QLabel("Docking Protocol Validation Using Decoy Compounds")
            title.setAlignment(Qt.AlignCenter)
            title.setStyleSheet("color:white;font-size:25px;font-weight:800;letter-spacing:0.2px;")
            sub=QLabel("Decoy-based docking-score discrimination • early enrichment • matched-decoy quality assessment")
            sub.setAlignment(Qt.AlignCenter)
            sub.setStyleSheet("color:#dbe8f6;font-size:11.5pt;font-weight:600;")
            dev=QLabel("Developed by Md Fazlay Rabbi  |  Department of Zoology, University of Rajshahi")
            dev.setAlignment(Qt.AlignCenter)
            dev.setStyleSheet("color:#bcd0e6;font-size:10.5pt;font-weight:600;")
            hl.addWidget(title); hl.addWidget(sub); hl.addWidget(dev); main.addWidget(header)
            files=QGroupBox("Input files"); gl=QGridLayout(files); self.file_labels={}
            entries=[("actives_scores","Putative actives docking scores"),("decoys_scores","Decoy docking scores"),("actives_smiles","Putative actives names + SMILES"),("decoys_smiles","Decoy names + SMILES")]
            for i,(key,text) in enumerate(entries):
                btn=QPushButton(text); btn.clicked.connect(lambda _,k=key:self.pick(k)); lab=QLabel("No file selected"); lab.setStyleSheet("color:#59677a;"); self.file_labels[key]=lab; r=(i//2)*2; c=(i%2)*2; gl.addWidget(btn,r,c); gl.addWidget(lab,r+1,c)
            main.addWidget(files)
            settings=QGroupBox("Analysis settings"); sl=QHBoxLayout(settings); self.target=QLineEdit("STAT3"); self.alpha=QDoubleSpinBox(); self.alpha.setRange(.1,100); self.alpha.setValue(20); self.boot=QSpinBox(); self.boot.setRange(500,50000); self.boot.setSingleStep(500); self.boot.setValue(5000); self.exp_leads=QSpinBox(); self.exp_leads.setRange(1,1000); self.exp_leads.setValue(10); self.exp_decoys=QSpinBox(); self.exp_decoys.setRange(1,10000); self.exp_decoys.setValue(50); self.strict=QCheckBox("Strict matched design"); self.strict.setChecked(True); self.lower=QCheckBox("Lower Vina score is better"); self.lower.setChecked(True)
            for text,w in [("Target",self.target),("BEDROC α",self.alpha),("Bootstrap",self.boot),("Expected leads",self.exp_leads),("Decoys / lead",self.exp_decoys)]: sl.addWidget(QLabel(text)); sl.addWidget(w)
            sl.addWidget(self.strict); sl.addWidget(self.lower); sl.addStretch()
            self.runbtn=QPushButton("Run Analysis")
            self.runbtn.setObjectName("run")
            self.runbtn.clicked.connect(self.run_gui)
            sl.addWidget(self.runbtn)
            main.addWidget(settings)

            # Output/export bar
            output_box=QGroupBox("Output files")
            ol=QHBoxLayout(output_box)
            out_note=QLabel("Export all files to your selected location")
            out_note.setStyleSheet("font-weight:700;color:#17324d;")
            self.output_path=QLineEdit()
            self.output_path.setReadOnly(True)
            self.output_path.setPlaceholderText("Choose a folder for publication figures, CSV tables, and reports")
            self.browseout=QPushButton("Choose Location")
            self.browseout.setObjectName("browseout")
            self.browseout.clicked.connect(self.choose_output_location)
            self.exportbtn=QPushButton("Export All Files")
            self.exportbtn.setObjectName("export")
            self.exportbtn.setEnabled(False)
            self.exportbtn.clicked.connect(self.export_gui)
            ol.addWidget(out_note)
            ol.addWidget(self.output_path, 1)
            ol.addWidget(self.browseout)
            ol.addWidget(self.exportbtn)
            main.addWidget(output_box)

            self.tabs=QTabWidget(); main.addWidget(self.tabs,1); self.overview=QTextEdit(); self.overview.setReadOnly(True); self.tabs.addTab(self.overview,"Overview")
            self.plot_tabs={}
            for title,key in [("Global summary","Global_discrimination_summary"),("ROC","ROC_curve"),("Precision–Recall","Precision_recall_curve"),("EF summary","Enrichment_metrics"),("Enrichment curve","Enrichment_curve"),("BEDROC","BEDROC_plot"),("Distributions","Score_distributions"),("Boxplot","Score_boxplot"),("Matched decoys","Matched_decoy_comparison"),("Decoy quality","Decoy_property_balance"),("Tanimoto","Decoy_Tanimoto_distribution")]:
                panel=PlotPanel(); self.plot_tabs[key]=panel; self.tabs.addTab(panel,title)
            self.table=QTableWidget(); self.tabs.addTab(self.table,"Matched-decoy table")
            self.log=QTextEdit(); self.log.setReadOnly(True); self.tabs.addTab(self.log,"Data / Log")
            self.progress=QProgressBar(); self.progress.setVisible(False); main.addWidget(self.progress)
            footer=QFrame(); footer.setStyleSheet("QFrame{background:#ffffff;border:1px solid #d7e0ea;border-radius:8px;}")
            fl=QHBoxLayout(footer); fl.setContentsMargins(12,7,12,7)
            self.status_label=QLabel("Ready")
            self.status_label.setStyleSheet("font-weight:700;color:#476176;")
            fl.addWidget(self.status_label); fl.addStretch()
            brand=QLabel("Docking Protocol Validation Using Decoy Compounds")
            brand.setStyleSheet("color:#708399;font-size:9.5pt;font-weight:600;")
            fl.addWidget(brand); main.addWidget(footer)
        def pick(self,key):
            p,_=QFileDialog.getOpenFileName(self,"Select CSV","","CSV files (*.csv);;All files (*)")
            if p: self.paths[key]=p; self.file_labels[key].setText(Path(p).name)
        def choose_output_location(self):
            d=QFileDialog.getExistingDirectory(self,"Choose output folder")
            if d:
                self.output_dir=d
                self.output_path.setText(d)
                self.status_label.setText("Output location selected")

        def run_gui(self):
            if not self.paths["actives_scores"] or not self.paths["decoys_scores"]:
                QMessageBox.warning(self,"Missing files","Load the two docking-score CSV files first."); return
            kwargs=dict(actives_csv=self.paths["actives_scores"],decoys_csv=self.paths["decoys_scores"],active_smiles_csv=self.paths["actives_smiles"] or None,decoy_smiles_csv=self.paths["decoys_smiles"] or None,lower_is_better=self.lower.isChecked(),bedroc_alpha=self.alpha.value(),n_bootstraps=self.boot.value(),random_seed=42,expected_leads=self.exp_leads.value(),expected_decoys_per_lead=self.exp_decoys.value(),strict_matched_design=self.strict.isChecked())
            self.runbtn.setEnabled(False); self.exportbtn.setEnabled(False); self.progress.setRange(0,0); self.progress.setVisible(True); self.log.setPlainText("Running analysis..."); self.status_label.setText("Analysis running..."); self.worker=AnalysisWorker(kwargs); self.worker.completed.connect(self.done); self.worker.failed.connect(self.failed); self.worker.start()
        def failed(self,msg):
            self.progress.setVisible(False); self.runbtn.setEnabled(True); self.status_label.setText("Analysis failed"); self.log.setPlainText(msg); QMessageBox.critical(self,"Analysis failed",msg[-3500:])
        def done(self,res):
            res.settings["target_name"] = self.target.text().strip() or "STAT3"
            self.result=res; self.progress.setVisible(False); self.runbtn.setEnabled(True); self.exportbtn.setEnabled(True); self.status_label.setText("Analysis completed — ready to export") ; self.overview.setPlainText(build_text_report(res)); self.log.setPlainText("Analysis completed successfully.\n\n"+json.dumps({"settings":res.settings,"matched_meta":res.matched_meta},indent=2,default=str)); figs=make_publication_figures(res)
            for key,panel in self.plot_tabs.items():
                if key in figs: panel.show_figure(figs[key])
            m=res.matched; self.table.clear(); self.table.setRowCount(len(m)); self.table.setColumnCount(len(m.columns)); self.table.setHorizontalHeaderLabels(list(m.columns))
            for i,row in m.iterrows():
                for j,col in enumerate(m.columns): self.table.setItem(i,j,QTableWidgetItem(str(row[col])))
            self.table.horizontalHeader().setSectionResizeMode(QHeaderView.ResizeToContents); QMessageBox.information(self,"Completed","Analysis completed successfully.")
        def export_gui(self):
            if self.result is None:
                return
            d=self.output_dir or self.output_path.text().strip()
            if not d:
                d=QFileDialog.getExistingDirectory(self,"Choose output folder")
                if not d:
                    return
                self.output_dir=d
                self.output_path.setText(d)
            target=re.sub(r"[^A-Za-z0-9_.-]+","_",self.target.text().strip() or "Target")
            out=Path(d)/f"{target}_Docking_Discrimination_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
            try:
                self.status_label.setText("Exporting publication package...")
                files=export_analysis_final(self.result,str(out),self.paths)
                self.status_label.setText(f"Export complete — {len(files)} files saved")
                self.output_path.setText(str(out))
                QMessageBox.information(self,"Export complete",f"Exported {len(files)} files to:\n{out}")
            except Exception:
                self.status_label.setText("Export failed")
                QMessageBox.critical(self,"Export failed",traceback.format_exc()[-4000:])

    app=QApplication([]); app.setApplicationName("Docking Protocol Validation Using Decoy Compounds"); w=App(); w.show(); app.exec_()


def _headless_self_test():
    import argparse
    ap=argparse.ArgumentParser(); ap.add_argument("--self-test",action="store_true"); ap.add_argument("--actives"); ap.add_argument("--decoys"); ap.add_argument("--active-smiles"); ap.add_argument("--decoy-smiles"); ap.add_argument("--out",default="DockingValidator_Test_Output"); ap.add_argument("--bootstraps",type=int,default=1000)
    args=ap.parse_args()
    r=run_analysis_final(args.actives,args.decoys,active_smiles_csv=args.active_smiles,decoy_smiles_csv=args.decoy_smiles,n_bootstraps=args.bootstraps,expected_leads=10,expected_decoys_per_lead=50,strict_matched_design=True)
    r.settings["target_name"] = "STAT3"
    export_analysis_final(r,args.out,{"actives_scores":args.actives,"decoys_scores":args.decoys,"actives_smiles":args.active_smiles,"decoys_smiles":args.decoy_smiles}); print(build_text_report(r)); print(f"\nExported to: {Path(args.out).resolve()}")


if __name__ == "__main__":
    import sys
    if "--self-test" in sys.argv:
        _headless_self_test()
    else:
        launch_gui_final()


The system cannot find the path specified.
